# Deep-layer lithography: into the resist volume

The quickstart stopped at the 2-D aerial image. Real thick-resist and
volumetric processes need the **z-resolved dose through the film**. This
notebook exercises three deep-layer paths exposed through the Python API:

1. **LIGA** - deep-X-ray shadow printing of hundreds of microns of PMMA.
2. **Multi-beam interference** - a standing-wave lattice recorded in one shot.
3. **Volumetric exposure + development** - a z-resolved latent image and the
   surface-relief depth map it develops into.

> **Maturity.** Every process here is **🔶 simplified** in
> `docs/capability-matrix.md` - separable/paraxial dose models, smoothed
> absorption edges, frozen-rate development. They run end to end and are
> physically motivated, but they are engineering approximations, not
> first-principles solvers. Treat the outputs as trends, not tape-out data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import highuvlith as huv
from highuvlith import api

%matplotlib inline

# These deep-layer bindings live on the native extension and are re-exported
# through highuvlith.api. If this import fails, rebuild with `maturin develop`.
print("simulate_liga        :", callable(api.simulate_liga))
print("simulate_interference:", callable(api.simulate_interference))
print("expose_volumetric    :", callable(api.expose_volumetric))

## 1. LIGA: deep-X-ray depth-dose

LIGA uses a bending-magnet white beam to shadow-print a gold absorber pattern
into very thick PMMA. Hard X-rays penetrate hundreds of microns, so the dose
falls only gradually with depth - that shallow gradient is what enables the
enormous aspect ratios LIGA is known for. `api.simulate_liga` returns the
depth-dose profile plus the **top/bottom dose ratio** and a damage-ceiling
flag.

In [ ]:
liga = api.simulate_liga(
    critical_energy_kev=6.23, resist_thickness_um=300.0,
    cd_nm=5000.0, pitch_nm=10000.0,
    grid_size=64, pixel_nm=200.0, nz=48,
)
z_um, dose = liga.depth_dose

plt.figure(figsize=(6, 4))
plt.plot(dose, z_um, color="darkgreen")
plt.gca().invert_yaxis()                      # surface (z=0) at the top
plt.xlabel("absorbed dose (kJ/cm$^3$)")
plt.ylabel("depth into PMMA (um)")
plt.title("LIGA depth-dose (300 um PMMA, E_c = 6.23 keV)")
plt.grid(alpha=0.3)

print(f"top dose        : {liga.top_dose_kj_cm3:.2f} kJ/cm^3")
print(f"bottom dose     : {liga.bottom_dose_kj_cm3:.2f} kJ/cm^3")
print(f"top:bottom ratio: {liga.dose_ratio:.2f}  (lower = more uniform)")
print(f"exceeds damage ceiling: {liga.exceeds_damage_ceiling}")

## 2. Multi-beam interference: a standing-wave lattice

Interference (holographic) lithography superposes a few coherent plane waves
into a periodic standing wave recorded as a PAC (photoactive-compound) volume
in a single exposure. For the `two_beam` preset the field is a 1-D grating
whose fringe period is fixed by the air-side half-angle. We slice the PAC
volume in the **x-z plane** to see the fringes run through the depth.

PAC convention: **m = 1 unexposed, m -> 0 fully exposed**, so bright fringes
(high dose) show up as low `m`.

In [ ]:
vol = api.simulate_interference(
    preset="two_beam", wavelength_nm=200.0, n_medium=1.6, half_angle_deg=30.0,
    nx=96, ny=8, nz=64,
    x_span_nm=1200.0, y_span_nm=100.0, z_span_nm=600.0,
    dose_scale=12.0,
)
pac = np.asarray(vol.values)                  # (nz, ny, nx)
xz = pac[:, pac.shape[1] // 2, :]             # slice at mid-y -> (nz, nx)
x_nm = np.asarray(vol.x_nm)
z_nm = np.asarray(vol.z_nm)

plt.figure(figsize=(7, 4))
im = plt.imshow(
    xz, origin="lower", aspect="auto", cmap="viridis",
    extent=[x_nm.min(), x_nm.max(), z_nm.min(), z_nm.max()],
)
plt.colorbar(im, label="PAC m (1 = unexposed)")
plt.xlabel("x (nm)")
plt.ylabel("z / depth (nm)")
plt.title("Two-beam interference: PAC in the x-z plane")
print(f"PAC range: {pac.min():.3f} (brightest fringe) .. {pac.max():.3f} (dark)")

## 3. Volumetric exposure + development

`api.expose_volumetric` builds the partially-coherent aerial image, refocuses
it through a stack of depth planes, folds in the exact transfer-matrix
intensity through the film stack, and integrates the Dill exposure with
split-step bleaching - giving a **z-resolved PAC volume**. Thresholding that
volume with `depth_map(...)` yields the surface-relief **development depth**
per column: spaces clear to the bottom, lines stay tall.

In [ ]:
source = huv.SourceConfig.f2_laser()
optics = huv.OpticsConfig(numerical_aperture=0.75)
mask = huv.MaskConfig.line_space(cd_nm=150, pitch_nm=300)
grid = huv.GridConfig(size=64, pixel_nm=5.0)
resist = huv.ResistConfig.vuv_fluoropolymer()

film = huv.FilmStackConfig()
film.add_layer("resist", 200.0, 1.5, 0.01)    # 200 nm resist, n = 1.5 + 0.01i
film.set_substrate(1.5, 1.0)                   # absorbing substrate

pac_vol = api.expose_volumetric(
    source, optics, mask, film, resist, grid,
    dose_mj_cm2=400.0, nz=40, n_defocus_planes=4, resist_layer=0, max_kernels=10,
)
depth = np.asarray(pac_vol.depth_map(0.5).values)   # (ny, nx), nm cleared
x_nm = np.asarray(pac_vol.x_nm)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 4))
im0 = ax0.imshow(
    depth, origin="lower", cmap="magma",
    extent=[x_nm.min(), x_nm.max(), x_nm.min(), x_nm.max()],
)
ax0.set(title="Developed depth map", xlabel="x (nm)", ylabel="y (nm)")
fig.colorbar(im0, ax=ax0, fraction=0.046, label="cleared depth (nm)")

ax1.plot(x_nm, depth[depth.shape[0] // 2], color="purple")
ax1.set(title="Depth profile (center row)", xlabel="x (nm)",
        ylabel="cleared depth (nm)")
ax1.grid(alpha=0.3)
fig.tight_layout()
print(f"cleared depth range: {depth.min():.0f} .. {depth.max():.0f} nm")

### API vs. CLI

Everything above uses the Python API. The same deep-layer processes are also
reachable from the command line, e.g.

```bash
cargo run -p highuvlith-cli -- deep --config examples/liga.toml
```

which is handy for batch runs and reproducible configs. See
`docs/processes/` for the physics write-ups and `docs/capability-matrix.md`
for the exact status of each model.